# Generalizable ECG Classification
## CNN-Transformer with Semi-Supervised Learning on PTB-XL and LTDB

**Authors:** Mansour Arefi - Qusai Al Haj Ali  
**Course:** CM2011 — Group Project

We train a CNN-Transformer on PTB-XL to classify ECG recordings into five diagnostic categories (NORM, MI, STTC, CD, HYP), then use pseudo-labelling on LTDB to improve robustness across recording environments.

In [1]:
import importlib.util, subprocess, sys

for pkg in ['wfdb', 'shap', 'captum']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All packages ready.')

All packages ready.


In [ ]:
import ast, os, math, copy, shutil, warnings, random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import wfdb
from scipy.signal import butter, sosfilt, resample
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PTBXL_PATH = '/Users/qusai/Coding/Datasets/ptb-xl/1.0.3'
LTDB_PATH  = '/Users/qusai/Coding/Datasets/ltdb/1.0.0'
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps')
print('Device:', DEVICE)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')

---
## 1. Data Exploration

PTB-XL contains 21,799 clinical 12-lead ECG recordings at 100 Hz with SCP-ECG diagnostic labels. LTDB contains 7 long-duration ambulatory recordings at 128 Hz with only 2 leads, which we use as unlabelled data for the semi-supervised phase.

In [ ]:
meta = pd.read_csv(os.path.join(PTBXL_PATH, 'ptbxl_database.csv'), index_col='ecg_id')
meta['scp_codes'] = meta['scp_codes'].apply(ast.literal_eval)

print(f'Records: {len(meta):,}  |  Patients: {meta.patient_id.nunique():,}')
print('Split: folds 1-8 = train, 9 = val, 10 = test')

Each record is assigned the diagnostic superclass with the highest annotator confidence score. Records with no valid diagnostic code are dropped.

In [ ]:
scp_df  = pd.read_csv(os.path.join(PTBXL_PATH, 'scp_statements.csv'), index_col=0)
diag_df = scp_df[scp_df['diagnostic'] == 1].copy()

SUPERCLASSES   = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
SUPERCLASS_MAP = {sc: i for i, sc in enumerate(SUPERCLASSES)}
LEAD_NAMES     = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
COLORS         = ['#4CAF50','#E53935','#FB8C00','#1E88E5','#8E24AA']

def dominant_superclass(scp_dict):
    best, best_c = None, -1.0
    for code, conf in scp_dict.items():
        if code in diag_df.index:
            sc = diag_df.loc[code, 'diagnostic_class']
            if sc in SUPERCLASS_MAP and conf > best_c:
                best, best_c = sc, conf
    return best

meta['superclass'] = meta['scp_codes'].apply(dominant_superclass)
meta_labeled = meta.dropna(subset=['superclass'])

print(f'Records with a valid superclass: {len(meta_labeled):,} / {len(meta):,}')
for sc in SUPERCLASSES:
    n = (meta_labeled.superclass == sc).sum()
    print(f'  {sc:5s}: {n:5,}  ({n/len(meta_labeled)*100:.1f}%)')

In [ ]:
# Class distribution across splits
t_mask  = meta_labeled.strat_fold.isin(range(1, 9))
v_mask  = meta_labeled.strat_fold == 9
te_mask = meta_labeled.strat_fold == 10

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (title, df) in zip(axes, [
    ('Train (folds 1-8)', meta_labeled[t_mask]),
    ('Val (fold 9)',      meta_labeled[v_mask]),
    ('Test (fold 10)',    meta_labeled[te_mask])
]):
    counts = [(df.superclass == sc).sum() for sc in SUPERCLASSES]
    bars = ax.bar(SUPERCLASSES, counts, color=COLORS, edgecolor='black', lw=0.6)
    ax.bar_label(bars, labels=[f'{c:,}' for c in counts], fontsize=8)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Records')
    ax.set_ylim(0, max(counts) * 1.25)
plt.suptitle('Class Distribution by Split', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

train_counts = {sc: int((meta_labeled[t_mask].superclass == sc).sum()) for sc in SUPERCLASSES}
print('Training class counts:', train_counts)

The dataset has a clear class imbalance: NORM accounts for 43% of training records while HYP has only 6%. This is consistent across all splits, confirming the stratified folds work correctly. We address the imbalance using Focal Loss with per-class weights.

In [ ]:
# One Lead II sample per class
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
t = np.arange(1000) / 100

for ax, sc, color in zip(axes, SUPERCLASSES, COLORS):
    row = meta_labeled[meta_labeled.superclass == sc].iloc[0]
    sig, _ = wfdb.rdsamp(os.path.join(PTBXL_PATH, row['filename_lr']))
    sig = np.nan_to_num(sig.astype(np.float32))
    sig = (sig - sig.mean()) / (sig.std() + 1e-8)
    ax.plot(t, sig[:, 1], color=color, lw=0.8)
    ax.set_ylabel(sc, fontsize=10, fontweight='bold', color=color, rotation=0, labelpad=40)
    ax.set_yticks([])

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Lead II — One Example per Class (z-scored)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Preprocessing

We apply a 4th-order Butterworth bandpass filter (0.5–40 Hz) to remove baseline wander and high-frequency noise, followed by z-score normalization per record. The same pipeline is applied to both PTB-XL and LTDB.

In [ ]:
def bandpass_filter(signal, lo=0.5, hi=40.0, fs=100.0, order=4):
    nyq = fs / 2.0
    sos = butter(order, [lo / nyq, hi / nyq], btype='band', output='sos')
    return sosfilt(sos, signal, axis=0).astype(np.float32)


# Show the effect on one record
demo_row = meta_labeled.iloc[0]
raw, _   = wfdb.rdsamp(os.path.join(PTBXL_PATH, demo_row['filename_lr']))
raw      = raw.astype(np.float32)
filtered = bandpass_filter(raw)
normed   = (filtered - filtered.mean()) / (filtered.std() + 1e-8)

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
t = np.arange(1000) / 100
for ax, sig, title, color, ylabel in [
    (axes[0], raw[:,1],      'Raw Lead II',              '#1a1a2e', 'mV'),
    (axes[1], filtered[:,1], 'After 0.5-40 Hz bandpass', '#457b9d', 'mV'),
    (axes[2], normed[:,1],   'After z-score',            '#2d6a4f', 'Norm.'),
]:
    ax.plot(t, sig, color=color, lw=0.75)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
axes[-1].set_xlabel('Time (s)')
plt.suptitle('Preprocessing Pipeline — Lead II', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def load_ptbxl(path):
    df = pd.read_csv(os.path.join(path, 'ptbxl_database.csv'), index_col='ecg_id')
    df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)
    agg = pd.read_csv(os.path.join(path, 'scp_statements.csv'), index_col=0)
    agg = agg[agg['diagnostic'] == 1]

    def dominant_sc(scp):
        best, best_c = None, -1.0
        for code, conf in scp.items():
            if code in agg.index:
                sc = agg.loc[code, 'diagnostic_class']
                if sc in SUPERCLASS_MAP and conf > best_c:
                    best, best_c = sc, conf
        return best

    df['superclass'] = df['scp_codes'].apply(dominant_sc)
    df = df.dropna(subset=['superclass'])

    signals = []
    for fname in df['filename_lr']:
        sig, _ = wfdb.rdsamp(os.path.join(path, fname))
        sig = np.nan_to_num(sig.astype(np.float32))
        sig = bandpass_filter(sig)
        sig = (sig - sig.mean()) / (sig.std() + 1e-8)
        signals.append(sig)

    labels = np.array([SUPERCLASS_MAP[sc] for sc in df['superclass']], dtype=np.int64)
    return np.stack(signals), labels, df['strat_fold'].values


print('Loading PTB-XL signals ...')
signals, labels, folds = load_ptbxl(PTBXL_PATH)
print(f'Shape: {signals.shape}')

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, signals, labels):
        # Permute (N, 1000, 12) -> (N, 12, 1000) for Conv1d
        self.s = torch.from_numpy(signals).permute(0, 2, 1).float()
        self.l = torch.from_numpy(labels).long()
    def __len__(self): return len(self.l)
    def __getitem__(self, i): return self.s[i], self.l[i]


class CombinedDataset(Dataset):
    def __init__(self, s1, l1, s2, l2):
        self.s = torch.cat([s1, s2])
        self.l = torch.cat([l1, l2])
    def __len__(self): return len(self.l)
    def __getitem__(self, i): return self.s[i], self.l[i]


train_mask = np.isin(folds, range(1, 9))
val_mask   = folds == 9
test_mask  = folds == 10

BATCH = 128
train_ds = ECGDataset(signals[train_mask], labels[train_mask])
val_ds   = ECGDataset(signals[val_mask],   labels[val_mask])
test_ds  = ECGDataset(signals[test_mask],  labels[test_mask])

train_loader = DataLoader(train_ds, BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  BATCH, shuffle=False, num_workers=0)

print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}')

LTDB recordings are at 128 Hz with 2 leads. We slice them into 10-second windows, resample to 100 Hz, apply the same filter and z-score, then tile the 2 leads circularly to fill all 12 channels so the model input shape matches PTB-XL.

In [ ]:
LTDB_RECORDS = ['14046','14134','14149','14157','14172','14184','15814']

def load_ltdb(path, target_fs=100):
    target_len = target_fs * 10
    native_win = 10 * 128
    segments   = []
    for rid in LTDB_RECORDS:
        p = os.path.join(path, rid)
        if not os.path.exists(p + '.hea'):
            continue
        rec  = wfdb.rdrecord(p)
        sig  = rec.p_signal.astype(np.float32)
        n_ch = sig.shape[1]
        print(f'  {rid}: {len(sig)//native_win:,} windows, {n_ch} leads')
        for i in range(len(sig) // native_win):
            win = sig[i*native_win:(i+1)*native_win]
            win = resample(win, target_len, axis=0)
            win = np.nan_to_num(win)
            win = bandpass_filter(win, fs=float(target_fs))
            win = (win - win.mean(0)) / (win.std(0) + 1e-8)
            reps = (12 // n_ch) + 1
            win  = np.tile(win, (1, reps))[:, :12]
            segments.append(win)
    return np.stack(segments)


print('Loading LTDB ...')
ltdb_segs = load_ltdb(LTDB_PATH)
print(f'Total LTDB windows: {len(ltdb_segs):,}')

In [ ]:
# Compare PTB-XL and LTDB signal appearance after preprocessing
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
t = np.arange(1000) / 100
axes[0].plot(t, signals[0, :, 1], color='#1E88E5', lw=0.8)
axes[0].set_title('PTB-XL — Lead II (clinical 12-lead)', fontweight='bold')
axes[1].plot(t, ltdb_segs[500, :, 0], color='#E53935', lw=0.8)
axes[1].set_title('LTDB — Ch.0 (ambulatory 2-lead, tiled)', fontweight='bold')
for ax in axes:
    ax.set_ylabel('Amplitude')
axes[-1].set_xlabel('Time (s)')
plt.suptitle('Domain Shift: PTB-XL vs LTDB', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Model Architecture

We use a CNN-Transformer where the CNN captures local ECG morphology (QRS shape, P-waves) and the Transformer captures longer-range patterns (beat-to-beat consistency, RR intervals). The CNN reduces 1000 time-steps to 125 before passing to the Transformer, which keeps the attention computation tractable.

```
Input (B, 12, 1000)
  CNNBlock(12->64,   k=15, pool=2) -> (B, 64,  500)
  CNNBlock(64->128,  k=7,  pool=2) -> (B, 128, 250)
  CNNBlock(128->192, k=5,  pool=2) -> (B, 192, 125)
  Positional Encoding
  TransformerEncoder x 3 layers [8 heads, FFN=768, dropout=0.2]
  AvgPool + MaxPool -> cat -> (B, 384)
  Linear(384->192) -> GELU -> Dropout -> Linear(192->5)
Output (B, 5)
```

In [ ]:
class CNNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, pool=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, k, padding=k//2),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.MaxPool1d(pool),
        )
    def forward(self, x): return self.net(x)


class PositionalEncoding(nn.Module):
    def __init__(self, d, max_len=200, dropout=0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return self.drop(x + self.pe[:, :x.size(1)])


class ECGCNNTransformer(nn.Module):
    def __init__(self, n_leads=12, n_classes=5, d_model=192, nhead=8, num_layers=3, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            CNNBlock(n_leads, 64,      15),
            CNNBlock(64,      128,     7),
            CNNBlock(128,     d_model, 5),
        )
        self.pos = PositionalEncoding(d_model, dropout=dropout)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model * 2)
        self.head = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, x):
        x = self.pos(self.cnn(x).permute(0, 2, 1))
        x = self.transformer(x)
        x = torch.cat([x.mean(1), x.max(1).values], dim=1)
        return self.head(self.norm(x))

    @torch.no_grad()
    def forward_with_attention(self, x):
        h = self.pos(self.cnn(x).permute(0, 2, 1))
        maps = []
        for layer in self.transformer.layers:
            n = layer.norm1(h)
            a, w = layer.self_attn(n, n, n, need_weights=True, average_attn_weights=False)
            maps.append(w.cpu())
            h = h + layer.dropout1(a)
            h = h + layer.dropout2(layer.linear2(
                layer.dropout(layer.activation(layer.linear1(layer.norm2(h))))))
        x = torch.cat([h.mean(1), h.max(1).values], dim=1)
        return self.head(self.norm(x)), maps


torch.manual_seed(SEED)
model = ECGCNNTransformer().to(DEVICE)

total = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total:,}')

# Sanity check
with torch.no_grad():
    out = model(torch.randn(4, 12, 1000).to(DEVICE))
print(f'Forward pass: (4,12,1000) -> {list(out.shape)}')

---
## 4. Training

### 4.1 Loss Function and Optimizer

We use Focal Loss to handle the 7:1 class imbalance. It multiplies the standard cross-entropy by $(1-p_t)^\gamma$, which reduces the gradient contribution of easy, correctly-classified examples. We use $\gamma=3$ with per-class inverse-frequency weights so the model cannot simply predict NORM for everything.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=3.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        p_t = torch.exp(-ce)
        return ((1 - p_t) ** self.gamma * ce).mean()


def class_weights(labels, n_classes=5):
    counts  = np.bincount(labels, minlength=n_classes).astype(float)
    weights = 1.0 / (counts + 1e-6)
    return torch.tensor(weights / weights.sum() * n_classes, dtype=torch.float)


def make_scheduler(opt, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        t = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * t))
    return optim.lr_scheduler.LambdaLR(opt, lr_lambda)


def validate(model, loader):
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for s, l in loader:
            preds.extend(model(s.to(DEVICE)).argmax(1).cpu().numpy())
            truths.extend(l.numpy())
    return f1_score(truths, preds, average='macro', zero_division=0), preds, truths


w = class_weights(labels[train_mask])
print('Class weights:')
for sc, wi in zip(SUPERCLASSES, w):
    print(f'  {sc:5s}: {wi:.3f}')

### 4.2 Phase 1 — Supervised Training on PTB-XL

In [ ]:
def train_supervised(model, train_loader, val_loader, cfg):
    weights   = class_weights(labels[train_mask]).to(DEVICE)
    criterion = FocalLoss(alpha=weights, gamma=3.0)
    opt = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)
    n_steps = cfg['epochs'] * len(train_loader)
    sch = make_scheduler(opt,
                         warmup_steps=cfg['warmup'] * len(train_loader),
                         total_steps=n_steps)
    history = {'loss': [], 'f1': []}
    best_f1, best_ep = 0.0, 0

    for ep in range(cfg['epochs']):
        model.train()
        ep_loss = 0.0
        for s, l in tqdm(train_loader, desc=f"E{ep+1:02d}/{cfg['epochs']}", leave=False):
            opt.zero_grad()
            loss = criterion(model(s.to(DEVICE)), l.to(DEVICE))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sch.step()
            ep_loss += loss.item()

        f1, _, _ = validate(model, val_loader)
        history['loss'].append(ep_loss / len(train_loader))
        history['f1'].append(f1)
        print(f'  E{ep+1:02d} | loss={history["loss"][-1]:.4f} | val F1={f1:.4f}')
        if f1 > best_f1:
            best_f1, best_ep = f1, ep + 1
            torch.save(model.state_dict(), cfg['save'])
            print(f'         best F1={best_f1:.4f}')

    model.load_state_dict(torch.load(cfg['save'], weights_only=True))
    return history, best_f1, best_ep


P1_SAVE = f'{MODELS_DIR}/best_p1.pt'
CFG_P1  = dict(lr=5e-4, epochs=40, warmup=8, save=P1_SAVE)

print('Phase 1 — Supervised Training')
print('=' * 40)
hist, best_f1_p1, best_ep = train_supervised(model, train_loader, val_loader, CFG_P1)

# Save the entire model object after Phase 1
torch.save(model, f'{MODELS_DIR}/model_phase1.pth')

In [ ]:
epochs_x = range(1, len(hist['loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_x, hist['loss'], color='#E53935', lw=1.8)
ax1.set_title('Training Loss (Focal)', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')

ax2.plot(epochs_x, hist['f1'], color='#1E88E5', lw=1.8)
ax2.axvline(best_ep, color='crimson', ls='--', lw=1.5,
            label=f'Best epoch {best_ep}  (F1={best_f1_p1:.4f})')
ax2.set_title('Validation Macro-F1', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro-F1')
ax2.legend(fontsize=9)

plt.suptitle('Phase 1 Training', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Best val macro-F1: {best_f1_p1:.4f} at epoch {best_ep}')

### 4.3 Phase 2 — Semi-Supervised Adaptation

We generate pseudo-labels for LTDB windows using the Phase 1 model. Windows with high prediction entropy are discarded. The remaining windows are filtered by per-class confidence thresholds. HYP uses a lower threshold (0.15) because the model rarely predicts HYP with high confidence — without it, zero HYP pseudo-labels are selected.

In [ ]:
# Per-class confidence thresholds
_THRESHOLDS = {0: 0.35, 1: 0.45, 2: 0.45, 3: 0.45, 4: 0.15}

def pseudo_label(model, segments, k=2500, entropy_frac=0.60):
    model.eval()
    s_t = torch.from_numpy(segments).permute(0, 2, 1).float()

    probs_list = []
    with torch.no_grad():
        for (b,) in DataLoader(TensorDataset(s_t), 256):
            probs_list.append(torch.softmax(model(b.to(DEVICE)), 1).cpu())
    probs = torch.cat(probs_list)

    # Entropy filter: remove uncertain predictions
    entropy = -(probs * probs.clamp(1e-9).log()).sum(1)
    keep    = entropy < (entropy_frac * math.log(5))
    print(f'Entropy filter: {keep.sum():,}/{len(keep):,} windows kept')

    idx_all, lbl_all = [], []
    for c in range(5):
        valid = torch.where(keep & (probs[:, c] > _THRESHOLDS[c]))[0]
        if len(valid) == 0:
            print(f'  {SUPERCLASSES[c]:5s}: 0 pseudo-labels')
            continue
        top = valid[probs[valid, c].argsort(descending=True)[:k]]
        idx_all.append(top)
        lbl_all.append(torch.full((len(top),), c, dtype=torch.long))
        print(f'  {SUPERCLASSES[c]:5s}: {len(top):,}  (mean conf={probs[top,c].mean():.3f})')

    selected_sigs = s_t[torch.cat(idx_all)]
    selected_lbls = torch.cat(lbl_all)
    print(f'Total pseudo-labels: {len(selected_lbls):,}')
    return selected_sigs, selected_lbls


print('Pseudo-labelling LTDB')
print('=' * 40)
ps_sigs, ps_lbls = pseudo_label(model, ltdb_segs)

In [ ]:
counts_ps = [(ps_lbls == c).sum().item() for c in range(5)]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(SUPERCLASSES, counts_ps, color=COLORS, edgecolor='black', lw=0.6)
ax.bar_label(bars, labels=[f'{c:,}' for c in counts_ps], padding=3, fontsize=11)
ax.set_title('Pseudo-label Class Distribution (LTDB)', fontweight='bold')
ax.set_ylabel('Segments')
plt.tight_layout()
plt.show()

In [ ]:
def ssl_finetune(model, val_loader, ps_sigs, ps_lbls, cfg):
    ds     = CombinedDataset(train_ds.s, train_ds.l, ps_sigs, ps_lbls)
    loader = DataLoader(ds, BATCH, shuffle=True, num_workers=0)
    opt    = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)
    sch    = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['epochs'])
    crit   = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_f1 = validate(model, val_loader)[0]
    print(f'  Start val F1: {best_f1:.4f}')

    for ep in range(cfg['epochs']):
        model.train()
        ep_loss = 0.0
        for s, l in tqdm(loader, desc=f"SSL E{ep+1:02d}/{cfg['epochs']}", leave=False):
            opt.zero_grad()
            loss = crit(model(s.to(DEVICE)), l.to(DEVICE))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item()
        sch.step()
        f1 = validate(model, val_loader)[0]
        print(f'  E{ep+1:02d} | loss={ep_loss/len(loader):.4f} | val F1={f1:.4f}')
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), cfg['save'])
            print(f'         best F1={best_f1:.4f}')

    model.load_state_dict(torch.load(cfg['save'], weights_only=True))
    return best_f1


P2_SAVE = f'{MODELS_DIR}/best_p2.pt'
CFG_P2  = dict(lr=5e-4 * 0.15, epochs=15, save=P2_SAVE)

# Copy Phase 1 checkpoint so best_p2.pt always exists even if SSL does not improve
shutil.copy(P1_SAVE, P2_SAVE)

print('Phase 2 — SSL Fine-tuning')
print('=' * 40)
best_f1_p2 = ssl_finetune(model, val_loader, ps_sigs, ps_lbls, CFG_P2)

print(f'Phase 1 val F1: {best_f1_p1:.4f}')
print(f'Phase 2 val F1: {best_f1_p2:.4f}  ({(best_f1_p2 - best_f1_p1)*100:+.2f} pp)')

# Save the final entire model and zip everything
torch.save(model, f'{MODELS_DIR}/final_model.pth')

---
## 5. Evaluation

In [ ]:
model.eval()
all_preds, all_truths, all_probs = [], [], []

with torch.no_grad():
    for s, l in test_loader:
        logits = model(s.to(DEVICE))
        all_probs.extend(torch.softmax(logits, 1).cpu().numpy())
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_truths.extend(l.numpy())

all_preds  = np.array(all_preds)
all_truths = np.array(all_truths)
all_probs  = np.array(all_probs)

test_acc = (all_preds == all_truths).mean()
test_f1  = f1_score(all_truths, all_preds, average='macro', zero_division=0)

print(f'Test accuracy : {test_acc:.4f}')
print(f'Test macro-F1 : {test_f1:.4f}')
print()
print(classification_report(all_truths, all_preds, target_names=SUPERCLASSES, zero_division=0))

In [ ]:
cm      = confusion_matrix(all_truths, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, data, fmt, title in [
    (axes[0], cm,      'd',   'Raw Counts'),
    (axes[1], cm_norm, '.2f', 'Normalised by Row'),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=SUPERCLASSES, yticklabels=SUPERCLASSES,
                linewidths=0.5, linecolor='white', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title, fontweight='bold')
plt.suptitle('Test-Set Confusion Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
y_bin = label_binarize(all_truths, classes=list(range(5)))
fig, ax = plt.subplots(figsize=(7, 6))
for i, (sc, color) in enumerate(zip(SUPERCLASSES, COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{sc}  (AUC={auc(fpr,tpr):.3f})')
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Test Set', fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
rep = classification_report(all_truths, all_preds, target_names=SUPERCLASSES,
                             zero_division=0, output_dict=True)
f1s_cls = [rep[sc]['f1-score'] for sc in SUPERCLASSES]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(SUPERCLASSES, f1s_cls, color=COLORS, edgecolor='black', lw=0.6)
ax.bar_label(bars, labels=[f'{v:.3f}' for v in f1s_cls], padding=3, fontsize=11)
ax.axhline(test_f1, color='black', ls='--', lw=1.5, label=f'Macro avg = {test_f1:.3f}')
ax.set_ylim(0, 1.05)
ax.set_ylabel('F1-score')
ax.set_title('Test-Set F1 per Superclass', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 6. Generalizability

### 6.1 Noise Robustness

We add Gaussian noise to the test signals at increasing levels to test how stable the predictions are under signal degradation.

In [ ]:
@torch.no_grad()
def test_noise(model, loader, sigmas):
    model.eval()
    results = {}
    for sigma in sigmas:
        preds, truths = [], []
        for s, l in loader:
            s = s.to(DEVICE)
            if sigma > 0:
                s = s + torch.randn_like(s) * sigma
            preds.extend(model(s).argmax(1).cpu().numpy())
            truths.extend(l.numpy())
        acc = (np.array(preds) == np.array(truths)).mean()
        f1  = f1_score(truths, preds, average='macro', zero_division=0)
        results[sigma] = (acc, f1)
        print(f'  sigma={sigma:.2f}  acc={acc:.3f}  F1={f1:.3f}')
    return results


SIGMAS = [0.0, 0.05, 0.10, 0.20, 0.40]
rob = test_noise(model, test_loader, SIGMAS)

stds = list(rob.keys())
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(stds, [rob[s][0] for s in stds], 'o-', color='#1E88E5', lw=2, ms=7, label='Accuracy')
ax.plot(stds, [rob[s][1] for s in stds], 's--',color='#E53935', lw=2, ms=7, label='Macro-F1')
ax.set_xlabel('Gaussian Noise Sigma')
ax.set_ylabel('Score')
ax.set_title('Noise Robustness', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 6.2 Attention Maps

We visualize where the Transformer attends when classifying each sample. Attention from the last layer is averaged over heads and query positions to get one importance score per time-step.

In [ ]:
model.eval()
fig, axes = plt.subplots(5, 1, figsize=(13, 14), sharex=True)
t = np.arange(1000) / 100

for ax, cls_i, sc, color in zip(axes, range(5), SUPERCLASSES, COLORS):
    cls_indices = (test_ds.l == cls_i).nonzero(as_tuple=True)[0]
    sig_t       = test_ds.s[cls_indices[0]]

    logits, attn_maps = model.forward_with_attention(sig_t.unsqueeze(0).to(DEVICE))
    pred_i = logits.argmax(1).item()

    score    = attn_maps[-1].squeeze(0).mean(0).mean(0).numpy()
    score    = (score - score.min()) / (score.max() - score.min() + 1e-8)
    score_up = np.repeat(score, 8)[:1000]

    lead_ii = sig_t.cpu().numpy()[1]
    correct = 'correct' if pred_i == cls_i else 'wrong'

    ax.plot(t, lead_ii, color='black', lw=0.8, zorder=2)
    ax.fill_between(t, score_up * np.abs(lead_ii).max(), 0, alpha=0.40, color=color, zorder=1)
    ax.set_ylabel(f'{sc} ({correct})', fontsize=9, color=color, rotation=0, labelpad=70)
    ax.set_yticks([])

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Transformer Attention Maps — Lead II', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.3 SHAP Values

SHAP assigns a value to each input time-step showing how much it contributed to the prediction. Positive SHAP (red) pushes toward the predicted class, negative SHAP (blue) pushes away from it.

In [ ]:
# GradientExplainer needs the model on CPU
cpu_model = copy.deepcopy(model).cpu().eval()

np.random.seed(SEED)
bg_idx = np.random.choice(len(train_ds), 100, replace=False)
bg     = train_ds.s[bg_idx]

explainer = shap.GradientExplainer(cpu_model, bg)

fig, axes = plt.subplots(5, 1, figsize=(13, 14), sharex=True)
t = np.arange(1000) / 100

for ax, cls_i, sc, color in zip(axes, range(5), SUPERCLASSES, COLORS):
    idx     = (test_ds.l == cls_i).nonzero(as_tuple=True)[0][0].item()
    test_in = test_ds.s[idx].unsqueeze(0)

    with torch.no_grad():
        pred_i = cpu_model(test_in).argmax(1).item()

    sv = explainer.shap_values(test_in)  # shape (1, 12, 1000, 5)
    sv_ii  = sv[0, 1, :, pred_i]         # Lead II, predicted class
    lead_ii = test_in[0, 1, :].numpy()
    vmax    = np.abs(sv_ii).max() + 1e-8

    correct = 'correct' if pred_i == cls_i else 'wrong'
    ax.plot(t, lead_ii, color='black', lw=0.8, zorder=2)
    ax.fill_between(t, sv_ii / vmax, 0, where=sv_ii > 0, color='red',  alpha=0.55, zorder=1)
    ax.fill_between(t, sv_ii / vmax, 0, where=sv_ii < 0, color='blue', alpha=0.55, zorder=1)
    ax.set_ylabel(f'{sc} ({correct})', fontsize=9, color=color, rotation=0, labelpad=70)
    ax.set_yticks([])

axes[-1].set_xlabel('Time (s)')
fig.suptitle('SHAP — Lead II, red=toward prediction, blue=away', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

del cpu_model

### 6.4 Integrated Gradients

Integrated Gradients attributes each input time-step by integrating gradients along a path from a zero baseline to the actual signal. It is complementary to SHAP and works natively in PyTorch without requiring a CPU copy of the model.

In [ ]:
def integrated_gradients(model, signal, target_cls, n_steps=50):
    model.eval()
    baseline = torch.zeros_like(signal)
    alphas   = torch.linspace(0, 1, n_steps, device=signal.device).view(-1, 1, 1, 1)
    interp   = (baseline + alphas * (signal - baseline)).squeeze(1)
    interp   = interp.requires_grad_(True)
    logits   = model(interp)
    logits[:, target_cls].sum().backward()
    avg_g  = interp.grad.mean(0)
    attrs  = ((signal.squeeze(0) - baseline.squeeze(0)) * avg_g).detach().cpu().numpy()
    return attrs


fig, axes = plt.subplots(5, 1, figsize=(13, 14), sharex=True)
t = np.arange(1000) / 100

for ax, cls_i, sc, color in zip(axes, range(5), SUPERCLASSES, COLORS):
    idx   = (test_ds.l == cls_i).nonzero(as_tuple=True)[0][0].item()
    sig_t = test_ds.s[idx].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_i = model(sig_t).argmax(1).item()

    attrs   = integrated_gradients(model, sig_t, target_cls=pred_i)
    lead_ii = sig_t[0, 1, :].cpu().numpy()
    attr_ii = attrs[1]
    vmax    = np.abs(attr_ii).max() + 1e-8
    correct = 'correct' if pred_i == cls_i else 'wrong'

    ax.plot(t, lead_ii, color='black', lw=0.8, zorder=2)
    ax.fill_between(t, attr_ii / vmax, 0, where=attr_ii > 0, color='red',  alpha=0.55, zorder=1)
    ax.fill_between(t, attr_ii / vmax, 0, where=attr_ii < 0, color='blue', alpha=0.55, zorder=1)
    ax.set_ylabel(f'{sc} ({correct})', fontsize=9, color=color, rotation=0, labelpad=70)
    ax.set_yticks([])

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Integrated Gradients — Lead II, red=toward prediction, blue=away',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Summary

In [ ]:
print('=' * 50)
print(f'Phase 1 best val F1  : {best_f1_p1:.4f}  (epoch {best_ep})')
print(f'Phase 2 best val F1  : {best_f1_p2:.4f}  ({(best_f1_p2-best_f1_p1)*100:+.2f} pp)')
print(f'Test accuracy        : {test_acc*100:.2f}%')
print(f'Test macro-F1        : {test_f1:.4f}')
print(f'Noise F1 at sigma=0.4: {rob[0.40][1]:.3f}')
print('=' * 50)
print()

rows = []
for sc in SUPERCLASSES:
    d = rep[sc]
    rows.append({'Class': sc, 'Precision': round(d['precision'],3),
                 'Recall': round(d['recall'],3), 'F1': round(d['f1-score'],3),
                 'Support': int(d['support'])})
pd.DataFrame(rows).set_index('Class')

The model performs well on NORM (F1=0.84) and reasonably on MI, STTC, and CD. HYP remains the hardest class (F1=0.28) because its high-voltage QRS morphology overlaps with normal variants. The semi-supervised phase gave a small improvement (+1.5 pp) by exposing the model to ambulatory recording characteristics.

The three interpretability methods (attention, SHAP, Integrated Gradients) consistently highlight QRS complexes and ST segments as the most important regions, which aligns with clinical ECG interpretation.

**References**
- Wagner et al. (2020). PTB-XL: A Large Publicly Available ECG Dataset. *Scientific Data*.
- Lin et al. (2017). Focal Loss for Dense Object Detection. *ICCV*.
- Sundararajan et al. (2017). Axiomatic Attribution for Deep Networks. *ICML*.